# **Pipeline NLP & Analisis Data: Skincare Lazada (200 Produk & Komentar)**
Notebook ini berisi tahapan lengkap mulai dari Web Scraping (simulasi terstruktur), Pre-processing Teks, Feature Engineering (TF-IDF), Text Similarity (Cosine Similarity), hingga Text Clustering (K-Means).

## **1. Instalasi Pustaka & Import Library**

In [ ]:
!pip install beautifulsoup4 selenium webdriver-manager pandas numpy nltk scikit-learn Sastrawi openpyxl

In [ ]:
import time
import re
import random
import pandas as pd
import numpy as np
import nltk
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

nltk.download('punkt')

## **2. Web Scraping / Pengumpulan Data (200 Produk Skincare, Merek, Harga, & Komentar)**
Bagian ini mensimulasikan hasil ekstraksi data dari halaman Lazada yang mencakup kolom Merek/Brand, Nama Produk, Harga, dan Komentar Konsumen.

In [ ]:
def generate_lazada_skincare_dataset(target_count=200):
    brands = [
        "Wardah", "Skintific", "Emina", "Scarlett Whitening", "Somethinc", 
        "Originote", "Azarine", "Glad2Glow", "NPure", "MS Glow", 
        "Implora", "Hada Labo", "Facetology", "Avoskin", "Garnier",
        "Bioaqua", "The Originote", "Hanasui", "CeraVe", "Pyunkang Yul"
    ]
    
    products = [
        "Moisturizer Gel", "Serum Brightening", "Sunscreen SPF 50", "Facial Wash Gentle", 
        "Toner Essence", "Clay Mask", "Sheet Mask", "Eye Cream", "Micellar Water", "Exfoliating Toner"
    ]
    
    comments = [
        "Produknya sangat bagus di kulit wajah saya yang kering dan langsung melembabkan seharian.",
        "Pengiriman cepat sekali, tapi packaging produk agak sedikit penyok saat diterima kurir.",
        "Serumnya bikin muka cerah bersinar dan tidak bikin breakout sama sekali di kulit sensitif.",
        "Kulit jadi lembab, kenyal, dan glowing setelah pemakaian rutin selama satu minggu.",
        "Kurang cocok di muka saya, setelah dipakai malah muncul jerawat kecil-kecil dan kemerahan.",
        "Harga sangat terjangkau dengan kualitas oke banget untuk mencerahkan wajah kusam.",
        "Teksturnya ringan, cepat meresap ke kulit, dan tidak terasa lengket sama sekali di muka.",
        "Sangat recommended buat yang punya masalah kulit kusam dan bernoda hitam bekas jerawat.",
        "Bikin wajah agak berminyak di siang hari, tapi wanginya enak dan menenangkan.",
        "Awalnya ragu beli di Lazada, ternyata produknya original dan hasilnya luar biasa memuaskan."
    ]
    
    data = []
    random.seed(42)
    
    for i in range(1, target_count + 1):
        brand = random.choice(brands)
        prod = random.choice(products)
        product_name = f"{brand} {prod}"
        price = random.randint(25, 300) * 1000
        comment = random.choice(comments)
        
        data.append({
            "No": i,
            "Merek/Brand": brand,
            "Nama Produk": product_name,
            "Harga (Rp)": price,
            "Komentar Konsumen": comment
        })
        
    return pd.DataFrame(data)

df_lazada = generate_lazada_skincare_dataset(200)
print(f"Total data berhasil dikumpulkan: {len(df_lazada)} baris produk.")
df_lazada.head(10)

## **3. Pre-processing Data Teks**
Tahapan pembersihan teks ulasan konsumen meliputi:
1. **Case Folding**: Mengubah seluruh teks menjadi huruf kecil.
2. **Cleaning**: Menghapus karakter khusus, angka, dan tanda baca.
3. **Stopword Removal**: Menghapus kata hubung/kata umum yang tidak bermakna penting.
4. **Stemming**: Mengubah kata berimbuhan ke bentuk dasarnya menggunakan Sastrawi.

In [ ]:
stemmer = StemmerFactory().create_stemmer()
stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())

def text_preprocessing(text):
    # 1. Case Folding
    text = text.lower()
    # 2. Cleaning (Hapus simbol & angka)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Tokenisasi & Stopword Removal
    tokens = text.split()
    tokens = [word for word in tokens if word not in stopwords]
    # 4. Stemming
    clean_text = ' '.join([stemmer.stem(word) for word in tokens])
    return clean_text

print("Menjalankan proses pre-processing teks ulasan...")
df_lazada['clean_komentar'] = df_lazada['Komentar Konsumen'].apply(text_preprocessing)
df_lazada[['Komentar Konsumen', 'clean_komentar']].head(10)

## **4. Feature Engineering (TF-IDF Vectorizer)**
Mengubah teks ulasan yang sudah bersih menjadi representasi matriks numerik berdasarkan bobot Term Frequency-Inverse Document Frequency.

In [ ]:
vectorizer = TfidfVectorizer(max_features=500)
X_tfidf = vectorizer.fit_transform(df_lazada['clean_komentar'])

print("Dimensi Matriks TF-IDF:", X_tfidf.shape)
print("Contoh Fitur Kata (Vocabulary) Teratas:")
print(list(vectorizer.get_feature_names_out())[:15])

## **5. Text Similarity (Cosine Similarity)**
Menghitung tingkat kemiripan antar ulasan konsumen untuk menemukan komentar yang paling mirip dengan ulasan tertentu.

In [ ]:
similarity_matrix = cosine_similarity(X_tfidf, X_tfidf)

print("Matriks Cosine Similarity (Ukuran 5x5 pertama):")
print(np.round(similarity_matrix[:5, :5], 3))

# Contoh pencarian 3 ulasan yang paling mirip dengan ulasan baris ke-0
target_idx = 0
similar_indices = similarity_matrix[target_idx].argsort()[::-1][1:4]

print(f"\nUlasan Referensi (Baris {target_idx}): \"{df_lazada['Komentar Konsumen'].iloc[target_idx]}\"")
print("--- 3 Ulasan Paling Mirip ---")
for idx in similar_indices:
    print(f"• Skor: {similarity_matrix[target_idx][idx]:.4f} | Produk: {df_lazada['Nama Produk'].iloc[idx]} | Teks: {df_lazada['Komentar Konsumen'].iloc[idx]}")

## **6. Text Clustering (K-Means Clustering)**
Mengelompokkan 200 ulasan produk skincare ke dalam 3 klaster utama (misalnya: Sentimen Positif Melembabkan/Mencerahkan, Masalah Iritasi/Breakout, dan Ulasan Logistik/Pengiriman).

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(X_tfidf)

df_lazada['cluster'] = kmeans.labels_

print("Distribusi Jumlah Data per Klaster:")
print(df_lazada['cluster'].value_counts())

for i in range(k):
    print(f"\n=== KLASTER {i} ===")
    samples = df_lazada[df_lazada['cluster'] == i][['Nama Produk', 'Komentar Konsumen']].head(3).values
    for s in samples:
        print(f"- [{s[0]}] : {s[1]}")